# Baseline2: Adding Finegrained Labels\

- after finishing the baseline-add imgtxt source idx can dot this work.

In [23]:
import os
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
    
# 注意路径
# /Users/suleynan_suir/Desktop/NLP_Package/Project/Hateful-Image-Project/data/all_data/info.csv
info_fp = '/Users/suleynan_suir/Desktop/NLP_Package/Project/Hateful-Image-Project/data/all_data/info.csv'

# 一定要改这个路径
# 额外下载的 citation 的数据文件
# /Users/suleynan_suir/Desktop/NLP_Package/Project/Hateful-Image-Project/data/all_data/hateful_memes_finegrained
fine_grained_data_dir = '/Users/suleynan_suir/Desktop/NLP_Package/Project/Hateful-Image-Project/data/all_data/hateful_memes_finegrained'

In [24]:
split = 'train'
# 额外数据标注集的是 json 类型
fp = f'{fine_grained_data_dir}/{split}.json'
fine_grained_df = pd.read_json(fp, lines=True)
fine_grained_df.head()

,id,set_name,img,text,gold_hate,gold_pc,gold_attack,pc,attack
0,42953,train,img/42953.png,its their character not their color that matters,[not_hateful],[pc_empty],[attack_empty],None,None
1,23058,train,img/23058.png,don't be afraid to love again everyone is not ...,[not_hateful],[pc_empty],[attack_empty],None,None
2,13894,train,img/13894.png,putting bows on your pet,[not_hateful],[pc_empty],[attack_empty],None,None
3,37408,train,img/37408.png,i love everything and everybody! except for sq...,[not_hateful],[pc_empty],[attack_empty],None,None
4,82403,train,img/82403.png,"everybody loves chocolate chip cookies, even h...",[not_hateful],[pc_empty],[attack_empty],None,None


In [25]:
fine_grained_df.tail()

,id,set_name,img,text,gold_hate,gold_pc,gold_attack,pc,attack
8495,10423,train,img/10423.png,nobody wants to hang auschwitz me,[hateful],[religion],[mocking],"[[religion], [religion], [religion]]","[[mocking], [mocking], [mocking]]"
8496,98203,train,img/98203.png,when god grants you a child after 20 years of ...,[hateful],[nationality],[dehumanizing],"[[nationality], [nationality], [religion]]","[[dehumanizing], [inciting_violence], []]"
8497,36947,train,img/36947.png,gays on social media: equality! body positivit...,[hateful],[sex],[exclusion],"[[sex], [sex], [sex]]","[[exclusion], [exclusion], [exclusion]]"
8498,16492,train,img/16492.png,having a bad day? you could be a siamese twin ...,[hateful],"[sex, disability]",[inferiority],"[[sex, disability], [sex, disability], [sex, d...","[[], [inferiority], [inferiority]]"
8499,15937,train,img/15937.png,i hate muslims too they take their religion to...,[hateful],[religion],"[inferiority, contempt]","[[religion], [religion], [religion]]","[[inferiority, contempt], [inferiority, contem..."


In [26]:
print(f'fine_grained_df.shape: {fine_grained_df.shape}')

fine_grained_df.shape: (8500, 9)


In [27]:
list(fine_grained_df.columns)

['id',
 'set_name',
 'img',
 'text',
 'gold_hate',
 'gold_pc',
 'gold_attack',
 'pc',
 'attack']

`gold` means: 机器学习和数据标注中，gold 一词通常用于指代"黄金标准"（Gold Standard），即被认为是最准确、最权威的标注或标签

In [28]:
print((fine_grained_df['gold_pc'].apply(lambda x: len(x)) > 1).sum())
print((fine_grained_df['gold_attack'].apply(lambda x: len(x)) > 1).sum())

389
327


- 标签二值化

In [29]:
mlb = MultiLabelBinarizer()
transformed = mlb.fit_transform(fine_grained_df['gold_attack'])
print(transformed.shape)
print(mlb.classes_)

(8500, 8)
['attack_empty' 'contempt' 'dehumanizing' 'exclusion' 'inciting_violence'
 'inferiority' 'mocking' 'slurs']


In [30]:
def find_unique_labels(lists, empty_replacement):
    
     if isinstance(lists, list):
          return list({item for lst in lists for item in lst})
     return [empty_replacement]
 
print((fine_grained_df['pc'].apply(find_unique_labels, empty_replacement='pc_empty').apply(lambda x: len(x)) > 1).sum())

print((fine_grained_df['attack'].apply(find_unique_labels, empty_replacement='attack_empty').apply(lambda x: len(x)) > 1).sum())

1077
1476


- 图像转化

分别两个类型的 label cols, 用两个不同的二值化工具来进行划分 (model fit得到：模型二值化多分类标签的能力)
> output: [xxx] one-hot encoding

```python
mlb_pc, mlb_attack = MultiLabelBinarizer(), MultiLabelBinarizer()
```

In [31]:
def find_unique_labels(lists, empty_replacement):
    
     if isinstance(lists, list):
          return list({item for lst in lists for item in lst})
     return [empty_replacement]
 

splits = ['train', 'dev_seen', 'dev_unseen']
# splits = ['train, ']

fine_grained_dfs = []

# fine_grained_data_dir = '../data/hateful_memes_finegrained'

for split in splits:
    fp = f'{fine_grained_data_dir}/{split}.json'
    fine_grained_df = pd.read_json(fp, lines=True)
    fine_grained_dfs.append(fine_grained_df)
    
fine_grained_df = pd.concat(fine_grained_dfs)

# 
mlb_pc, mlb_attack = MultiLabelBinarizer(), MultiLabelBinarizer()
mlb_pc.fit(fine_grained_df['gold_pc'])
mlb_attack.fit(fine_grained_df['gold_attack'])

print(type(mlb_pc))
print(type(mlb_attack))
# mlb_pc / mlb_attack 这两个是 model对象

<class 'sklearn.preprocessing._label.MultiLabelBinarizer'>
<class 'sklearn.preprocessing._label.MultiLabelBinarizer'>


In [32]:
splits = ['train', 'dev_seen', 'dev_unseen']
fine_grained_dfs = []

for split in splits:
    fp = f'{fine_grained_data_dir}/{split}.json'
    fine_grained_df = pd.read_json(fp, lines=True)
    # transform 'pc' like 'gold_pc'
    fine_grained_df['pc'] = fine_grained_df['pc'].apply(find_unique_labels, empty_replacement='pc_empty')
    # transform 'attack' like 'gold_attack'
    fine_grained_df['attack'] = fine_grained_df['attack'].apply(find_unique_labels, empty_replacement='attack_empty')
    # binarize 'gold_pc' and 'gold_attack'
    new_cols = [x+'_gold_pc' for x in mlb_pc.classes_]
    fine_grained_df[new_cols] = mlb_pc.transform(fine_grained_df['gold_pc'])
    new_cols = [x+'_gold_attack' for x in mlb_attack.classes_]
    fine_grained_df[new_cols] = mlb_attack.transform(fine_grained_df['gold_attack'])
    # binarize 'pc' and 'attack'
    new_cols = [x+'_pc' for x in mlb_pc.classes_]
    fine_grained_df[new_cols] = mlb_pc.transform(fine_grained_df['pc'])
    new_cols = [x+'_attack' for x in mlb_attack.classes_]
    fine_grained_df[new_cols] = mlb_attack.transform(fine_grained_df['attack'])

    fine_grained_dfs.append(fine_grained_df)

fine_grained_df = pd.concat(fine_grained_dfs)
print(fine_grained_df.shape)

fine_grained_df.head()

(9540, 37)


/opt/anaconda3/envs/mlenv/lib/python3.13/site-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['class'] will be ignored
  warnings.warn(


,id,set_name,img,text,gold_hate,gold_pc,gold_attack,pc,attack,disability_gold_pc,...,religion_pc,sex_pc,attack_empty_attack,contempt_attack,dehumanizing_attack,exclusion_attack,inciting_violence_attack,inferiority_attack,mocking_attack,slurs_attack
0,42953,train,img/42953.png,its their character not their color that matters,[not_hateful],[pc_empty],[attack_empty],[pc_empty],[attack_empty],0,...,0,0,1,0,0,0,0,0,0,0
1,23058,train,img/23058.png,don't be afraid to love again everyone is not ...,[not_hateful],[pc_empty],[attack_empty],[pc_empty],[attack_empty],0,...,0,0,1,0,0,0,0,0,0,0
2,13894,train,img/13894.png,putting bows on your pet,[not_hateful],[pc_empty],[attack_empty],[pc_empty],[attack_empty],0,...,0,0,1,0,0,0,0,0,0,0
3,37408,train,img/37408.png,i love everything and everybody! except for sq...,[not_hateful],[pc_empty],[attack_empty],[pc_empty],[attack_empty],0,...,0,0,1,0,0,0,0,0,0,0
4,82403,train,img/82403.png,"everybody loves chocolate chip cookies, even h...",[not_hateful],[pc_empty],[attack_empty],[pc_empty],[attack_empty],0,...,0,0,1,0,0,0,0,0,0,0


In [33]:
list(fine_grained_df.columns)

['id',
 'set_name',
 'img',
 'text',
 'gold_hate',
 'gold_pc',
 'gold_attack',
 'pc',
 'attack',
 'disability_gold_pc',
 'nationality_gold_pc',
 'pc_empty_gold_pc',
 'race_gold_pc',
 'religion_gold_pc',
 'sex_gold_pc',
 'attack_empty_gold_attack',
 'contempt_gold_attack',
 'dehumanizing_gold_attack',
 'exclusion_gold_attack',
 'inciting_violence_gold_attack',
 'inferiority_gold_attack',
 'mocking_gold_attack',
 'slurs_gold_attack',
 'disability_pc',
 'nationality_pc',
 'pc_empty_pc',
 'race_pc',
 'religion_pc',
 'sex_pc',
 'attack_empty_attack',
 'contempt_attack',
 'dehumanizing_attack',
 'exclusion_attack',
 'inciting_violence_attack',
 'inferiority_attack',
 'mocking_attack',
 'slurs_attack']

In [34]:
cols_to_remove = ['img', 'text', 'gold_hate']
fine_grained_df = fine_grained_df.drop(columns=cols_to_remove)
fine_grained_df = fine_grained_df.rename(columns={'set_name':'split'})
print(fine_grained_df.shape)

fine_grained_df.head()

(9540, 34)


,id,split,gold_pc,gold_attack,pc,attack,disability_gold_pc,nationality_gold_pc,pc_empty_gold_pc,race_gold_pc,...,religion_pc,sex_pc,attack_empty_attack,contempt_attack,dehumanizing_attack,exclusion_attack,inciting_violence_attack,inferiority_attack,mocking_attack,slurs_attack
0,42953,train,[pc_empty],[attack_empty],[pc_empty],[attack_empty],0,0,1,0,...,0,0,1,0,0,0,0,0,0,0
1,23058,train,[pc_empty],[attack_empty],[pc_empty],[attack_empty],0,0,1,0,...,0,0,1,0,0,0,0,0,0,0
2,13894,train,[pc_empty],[attack_empty],[pc_empty],[attack_empty],0,0,1,0,...,0,0,1,0,0,0,0,0,0,0
3,37408,train,[pc_empty],[attack_empty],[pc_empty],[attack_empty],0,0,1,0,...,0,0,1,0,0,0,0,0,0,0
4,82403,train,[pc_empty],[attack_empty],[pc_empty],[attack_empty],0,0,1,0,...,0,0,1,0,0,0,0,0,0,0


In [35]:
fine_grained_df.columns

Index(['id', 'split', 'gold_pc', 'gold_attack', 'pc', 'attack',
       'disability_gold_pc', 'nationality_gold_pc', 'pc_empty_gold_pc',
       'race_gold_pc', 'religion_gold_pc', 'sex_gold_pc',
       'attack_empty_gold_attack', 'contempt_gold_attack',
       'dehumanizing_gold_attack', 'exclusion_gold_attack',
       'inciting_violence_gold_attack', 'inferiority_gold_attack',
       'mocking_gold_attack', 'slurs_gold_attack', 'disability_pc',
       'nationality_pc', 'pc_empty_pc', 'race_pc', 'religion_pc', 'sex_pc',
       'attack_empty_attack', 'contempt_attack', 'dehumanizing_attack',
       'exclusion_attack', 'inciting_violence_attack', 'inferiority_attack',
       'mocking_attack', 'slurs_attack'],
      dtype='object')

- 连接前面处理后的相关联性数据

In [36]:
# info_fp = '../data/hateful_memes/info.csv'

info_df = pd.read_csv(info_fp)
print(info_df.shape)
info_df.head()

(10000, 8)


,id,img,label,text,split,text_idx,pseudo_text_idx,pseudo_img_idx
0,42953,img/42953.png,0.0,its their character not their color that matters,train,0,2161,0
1,23058,img/23058.png,0.0,don't be afraid to love again everyone is not ...,train,1,2931,2816
2,13894,img/13894.png,0.0,putting bows on your pet,train,2,0,5188
3,37408,img/37408.png,0.0,i love everything and everybody! except for sq...,train,3,3542,3435
4,82403,img/82403.png,0.0,"everybody loves chocolate chip cookies, even h...",train,4,4614,1


In [37]:
# merge
info_df = pd.merge(info_df, fine_grained_df, on=['id', 'split'], how='left')

print(info_df.shape)
info_df.tail()

(10000, 40)


,id,img,label,text,split,text_idx,pseudo_text_idx,pseudo_img_idx,gold_pc,gold_attack,...,religion_pc,sex_pc,attack_empty_attack,contempt_attack,dehumanizing_attack,exclusion_attack,inciting_violence_attack,inferiority_attack,mocking_attack,slurs_attack
9995,3869,img/03869.png,NaN,a mother's love for the child is a divine thing,test,8041,3378,3293,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9996,23817,img/23817.png,NaN,sea monkeys,test,1103,367,2926,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9997,56280,img/56280.png,NaN,little miss muffet sat on her tuffet,test,8042,7229,7102,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9998,29384,img/29384.png,NaN,they're in a row,test,8043,3724,1609,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9999,34127,img/34127.png,NaN,that feeling when you win a fifa game after be...,test,8044,4798,1632,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
# 保证所有 idx 的数值类型都是 int
float_cols = info_df.select_dtypes(float).columns
info_df[float_cols] = info_df.select_dtypes(float).astype('Int64')
info_df.head()

,id,img,label,text,split,text_idx,pseudo_text_idx,pseudo_img_idx,gold_pc,gold_attack,...,religion_pc,sex_pc,attack_empty_attack,contempt_attack,dehumanizing_attack,exclusion_attack,inciting_violence_attack,inferiority_attack,mocking_attack,slurs_attack
0,42953,img/42953.png,0,its their character not their color that matters,train,0,2161,0,[pc_empty],[attack_empty],...,0,0,1,0,0,0,0,0,0,0
1,23058,img/23058.png,0,don't be afraid to love again everyone is not ...,train,1,2931,2816,[pc_empty],[attack_empty],...,0,0,1,0,0,0,0,0,0,0
2,13894,img/13894.png,0,putting bows on your pet,train,2,0,5188,[pc_empty],[attack_empty],...,0,0,1,0,0,0,0,0,0,0
3,37408,img/37408.png,0,i love everything and everybody! except for sq...,train,3,3542,3435,[pc_empty],[attack_empty],...,0,0,1,0,0,0,0,0,0,0
4,82403,img/82403.png,0,"everybody loves chocolate chip cookies, even h...",train,4,4614,1,[pc_empty],[attack_empty],...,0,0,1,0,0,0,0,0,0,0


In [39]:
# info_fine_grained label
info_df.to_csv(info_fp.replace('info', 'info_fine_grained'), index=False)

pc_columns = [col for col in info_df.columns if col.endswith('_pc') and not 'gold' in col]
attack_columns = [col for col in info_df.columns if col.endswith('_attack') and not 'gold' in col]
fine_grained_labels = pc_columns + attack_columns
print(fine_grained_labels)

['disability_pc', 'nationality_pc', 'pc_empty_pc', 'race_pc', 'religion_pc', 'sex_pc', 'attack_empty_attack', 'contempt_attack', 'dehumanizing_attack', 'exclusion_attack', 'inciting_violence_attack', 'inferiority_attack', 'mocking_attack', 'slurs_attack']


In [40]:
with open('fine_grained_labels.txt', 'w') as file:
    file.writelines([line+'\n' for line in fine_grained_labels])